# Benchmark temporel (TensorFlow/Keras) — **LSTM · TCN · TFT** — prévision du **Heat Index** (NOAA GSOD, stations Maroc)

Version **comparaison** de `train_models.ipynb` : on entraîne et compare **trois**
architectures temporelles — **LSTM**, **TCN** (convolutions causales dilatées) et
**TFT** (Temporal Fusion Transformer compact) — sur la même cible **heat_index**.

À partir des `INPUT_WINDOW` derniers jours de variables météo, chaque modèle prédit
les `OUTPUT_WINDOW` prochains jours du **heat_index** ; interface commune
`(INPUT_WINDOW, n_features) -> (OUTPUT_WINDOW,)`, donc directement comparables.

Pipeline : chargement `data/cleandata_gsod_by_station` → nettoyage + filtre
saisonnier (avril→sept) → **heatmap de corrélation** → split chronologique par
station 70/15/15 → normalisation Min-Max (fit train only) → fenêtrage → **benchmark
3 modèles Keras** → évaluation (RMSE/MAE/R²) → **comparaison** → visualisation →
sauvegarde (`.keras` + scalers).

## 0. Imports & configuration

In [ ]:
import os, glob, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings; warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ── Configuration centrale (source de vérité unique) ──
from dataclasses import dataclass, field
from typing import List

@dataclass
class Config:
    """Hyperparamètres, chemins, features et cible du benchmark (NOAA GSOD).
    Modifier ICI se propage à tout le notebook."""
    # — Données / chemins —
    data_dir: str = ""            # résolu automatiquement ci-dessous
    art_dir:  str = "artifacts_compare"
    # — Features & cible (sélectionnées PAR NOM) —
    features: List[str] = field(default_factory=lambda: [
        "latitude", "longitude", "elevation", "distance_to_ocean_km",
        "tmax_c", "tmin_c", "tmean_c", "dewp_c", "humidity_pct",
        "precip_mm", "wind_speed_ms", "press_hpa",
    ])
    target: str = "heat_index"
    # — Modèles comparés —
    model_names: List[str] = field(default_factory=lambda: ["LSTM", "TCN", "TFT"])
    # — Filtre saisonnier (mois inclus, 1-12) : 4->9 = avril à septembre —
    month_start: int = 4
    month_end:   int = 9
    # — Fenêtrage temporel —
    input_window:  int = 7        # jours passés fournis au modèle
    output_window: int = 7        # jours prédits
    # — Split chronologique par station (70 / 15 / 15) —
    train_ratio: float = 0.70
    val_ratio:   float = 0.15
    test_ratio:  float = 0.15
    # — Modèle —
    hidden:   int = 128
    n_layers: int = 2
    dropout:  float = 0.2
    # — Entraînement —
    batch_size: int = 256
    epochs:     int = 30
    lr:         float = 1e-3
    patience:   int = 8
    seed:       int = 42

    @property
    def n_features(self) -> int: return len(self.features)

cfg = Config()

# Résolution robuste du dossier de données (relatif au repo, peu importe où tourne le kernel)
_here = Path.cwd()
_candidates = [
    Path(cfg.data_dir) if cfg.data_dir else None,
    _here / "data" / "cleandata_gsod_by_station",
    _here.parent / "data" / "cleandata_gsod_by_station",
    _here.parent.parent / "data" / "cleandata_gsod_by_station",
]
DATA_DIR = next((str(p) for p in _candidates if p and Path(p).is_dir()), None)
assert DATA_DIR, "Dossier data/cleandata_gsod_by_station introuvable"
cfg.data_dir = DATA_DIR

ART_DIR = Path(cfg.art_dir); ART_DIR.mkdir(exist_ok=True)

# Alias courts (utilisés dans tout le notebook)
FEATURES, TARGET   = cfg.features, cfg.target
MODEL_NAMES        = cfg.model_names
MONTH_START, MONTH_END = cfg.month_start, cfg.month_end
INPUT_WINDOW, OUTPUT_WINDOW = cfg.input_window, cfg.output_window
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = cfg.train_ratio, cfg.val_ratio, cfg.test_ratio
HIDDEN, N_LAYERS, DROPOUT = cfg.hidden, cfg.n_layers, cfg.dropout
BATCH_SIZE, EPOCHS, LR, PATIENCE = cfg.batch_size, cfg.epochs, cfg.lr, cfg.patience

print("DATA_DIR:", DATA_DIR)
print("modèles :", MODEL_NAMES)
print("features:", FEATURES)
print("target  :", TARGET)

## 1. Chargement de toutes les stations

In [ ]:
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
assert files, f"Aucun CSV dans {DATA_DIR}"
frames = []
for f in files:
    df = pd.read_csv(f, parse_dates=["date"])
    df["station"] = Path(f).stem
    frames.append(df)
data = pd.concat(frames, ignore_index=True)
print("stations:", data["station"].nunique(), "| lignes:", len(data))
print("période:", data["date"].min().date(), "->", data["date"].max().date())
data.head()

## 2. Nettoyage & filtre saisonnier (avril → septembre)
Tri chronologique par station, **filtrage sur la saison chaude** (mois
`MONTH_START`..`MONTH_END`, par défaut 4→9 — période pertinente pour le heat index)
puis suppression des lignes ayant un NaN dans une colonne utilisée (features + cible).

In [ ]:
def filter_months(df, start=MONTH_START, end=MONTH_END):
    """Ne garde que les lignes dont le mois est dans [start, end] (bornes incluses).
    Gère aussi les plages qui traversent le nouvel an (ex. start=11, end=2)."""
    m = df["date"].dt.month
    mask = (m >= start) & (m <= end) if start <= end else (m >= start) | (m <= end)
    return df[mask]

data = data.sort_values(["station", "date"]).reset_index(drop=True)

before = len(data)
data = filter_months(data).reset_index(drop=True)
print(f"filtre mois {MONTH_START}->{MONTH_END} : {before} -> {len(data)} lignes")

need = FEATURES + [TARGET]
before = len(data)
data = data.dropna(subset=need).reset_index(drop=True)
print(f"supprimé {before - len(data)} lignes NaN -> {len(data)} lignes")
data[need].describe().round(2)

## 2bis. Matrice de corrélation (features & cible)
Heatmap de Pearson (seaborn) sur le jeu complet **après** nettoyage — pour repérer
la redondance entre features et les liens avec la cible `heat_index` avant
l'entraînement.

In [ ]:
# ── Heatmap de corrélation complète (features + cible) ──
corr_cols = FEATURES + [TARGET]
corr = data[corr_cols].corr()

plt.figure(figsize=(13, 11))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            vmin=-1, vmax=1, square=True, linewidths=.5, cbar_kws={"shrink": .8})
plt.title("Matrice de corrélation — features & heat_index", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

# Paires fortement colinéaires (|r| > 0.9, hors diagonale) -> redondance potentielle
keep = np.tril(np.ones_like(corr, dtype=bool), k=-1)
pairs = corr.where(keep).abs().stack().sort_values(ascending=False)
pairs = pairs[pairs > 0.9]
print("Paires |r| > 0.9 (redondance potentielle):")
for (a, b), _ in pairs.items():
    print(f"  {a:>20s} ~ {b:<20s} r={corr.loc[a, b]:+.3f}")

# Top corrélations (|r|) de la cible avec les features
order = corr[TARGET].drop(TARGET).abs().sort_values(ascending=False).head(8).index
print(f"\nTop |r| avec {TARGET}:")
print(corr.loc[order, TARGET].round(3).to_string())

## 3. Split chronologique par station — 70 / 15 / 15
On découpe **chaque station** dans l'ordre temporel : 70 % (train), puis 15 %
(val), puis 15 % (test). Pas de mélange aléatoire, pas de chevauchement de stations
entre les jeux — chaque jeu reste contigu dans le temps (évite la fuite de données).

In [ ]:
def split_station(g):
    g = g.sort_values("date")
    n = len(g)
    i_tr = int(n * TRAIN_RATIO)
    i_va = int(n * (TRAIN_RATIO + VAL_RATIO))
    return g.iloc[:i_tr], g.iloc[i_tr:i_va], g.iloc[i_va:]

parts = [split_station(g) for _, g in data.groupby("station")]
train_df = pd.concat([p[0] for p in parts], ignore_index=True)
val_df   = pd.concat([p[1] for p in parts], ignore_index=True)
test_df  = pd.concat([p[2] for p in parts], ignore_index=True)

tot = len(data)
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s}: {len(d):7d} lignes ({len(d)/tot*100:4.1f}%)  "
          f"{d['date'].min().date()} -> {d['date'].max().date()}")

## 4. Normalisation Min-Max (fit sur TRAIN uniquement)
`MinMaxScaler` sur les features et sur la cible, ajusté **seulement** sur le train
(pas de fuite). Le scaler de la cible est conservé pour l'inverse-transform.

In [ ]:
scaler_X = MinMaxScaler().fit(train_df[FEATURES])
scaler_y = MinMaxScaler().fit(train_df[[TARGET]])

def scale(df):
    df = df.copy()
    df[FEATURES] = scaler_X.transform(df[FEATURES])
    df[TARGET]   = scaler_y.transform(df[[TARGET]])
    return df

train_n, val_n, test_n = scale(train_df), scale(val_df), scale(test_df)
print("features range (train, post-scale):",
      float(train_n[FEATURES].min().min()), "->", float(train_n[FEATURES].max().max()))
print(f"{TARGET} data_min_ / data_max_:",
      round(float(scaler_y.data_min_[0]), 2), "/", round(float(scaler_y.data_max_[0]), 2))

## 5. Fenêtrage — séquences glissantes par station
Les fenêtres ne traversent jamais une frontière de station. X = `INPUT_WINDOW`
jours de features ; y = les `OUTPUT_WINDOW` jours suivants de la cible.

In [ ]:
def make_sequences(df_norm):
    Xs, Ys = [], []
    win = INPUT_WINDOW + OUTPUT_WINDOW
    for _, g in df_norm.groupby("station"):
        g = g.sort_values("date")
        F = g[FEATURES].to_numpy(np.float32)
        T = g[TARGET].to_numpy(np.float32)
        for i in range(len(g) - win + 1):
            Xs.append(F[i:i+INPUT_WINDOW])
            Ys.append(T[i+INPUT_WINDOW:i+win])
    if not Xs:
        return (np.empty((0, INPUT_WINDOW, len(FEATURES)), np.float32),
                np.empty((0, OUTPUT_WINDOW), np.float32))
    return np.stack(Xs), np.stack(Ys)

X_tr, y_tr = make_sequences(train_n)
X_va, y_va = make_sequences(val_n)
X_te, y_te = make_sequences(test_n)
print("X_train:", X_tr.shape, "y_train:", y_tr.shape)
print("X_val  :", X_va.shape, "| X_test:", X_te.shape)

## 6. Modèles temporels Keras — **LSTM · TCN · TFT**
Trois architectures partageant la même interface
`(INPUT_WINDOW, n_features) -> (OUTPUT_WINDOW,)`, donc directement comparables :
- **LSTM** — RNN empilé (`N_LAYERS` couches).
- **TCN** — convolutions 1D causales dilatées (non-récurrent).
- **TFT** — Temporal Fusion Transformer compact (GRN + encodeur LSTM + self-attention multi-tête).

In [ ]:
def _grn(x, units, dropout=DROPOUT):
    """Gated Residual Network (Lim et al., 2021) — brique de base du TFT.
    Projection ELU -> gating GLU -> connexion résiduelle -> LayerNorm."""
    residual = x if x.shape[-1] == units else layers.Dense(units)(x)
    h = layers.Dense(units, activation="elu")(x)
    h = layers.Dense(units)(h)
    h = layers.Dropout(dropout)(h)
    gate = layers.Dense(units, activation="sigmoid")(h)   # GLU
    h = layers.Multiply()([h, gate])
    return layers.LayerNormalization()(layers.Add()([residual, h]))


def build_model(name):
    """Construit & compile un modèle Keras (cible unique) à partir de la config."""
    inp = keras.Input(shape=(INPUT_WINDOW, len(FEATURES)))

    if name == "LSTM":
        x = inp
        for li in range(N_LAYERS):
            x = layers.LSTM(HIDDEN, return_sequences=(li < N_LAYERS - 1))(x)
            x = layers.Dropout(DROPOUT)(x)
    elif name == "TCN":
        x = inp
        for d in (1, 2, 4):                         # convolutions causales dilatées
            x = layers.Conv1D(HIDDEN, 3, padding="causal", dilation_rate=d,
                              activation="relu")(x)
            x = layers.Dropout(DROPOUT)(x)
        x = x[:, -1, :]                             # dernier pas de temps
    elif name == "TFT":
        # Temporal Fusion Transformer (version compacte) : embedding par variable
        # via GRN -> encodeur LSTM -> self-attention multi-tête -> GRN.
        u = HIDDEN
        x = layers.Dense(u)(inp)                    # embedding des variables
        x = _grn(x, u)                              # variable selection / processing
        x = layers.LSTM(u, return_sequences=True)(x)  # traitement local séquentiel
        x = _grn(x, u)                              # enrichissement
        attn = layers.MultiHeadAttention(num_heads=4, key_dim=max(1, u // 4))(x, x)
        x = layers.LayerNormalization()(layers.Add()([x, layers.Dropout(DROPOUT)(attn)]))
        x = _grn(x, u)                              # feed-forward position-wise
        x = x[:, -1, :]                             # contexte final
    else:
        raise ValueError(f"modèle inconnu: {name}")

    out = layers.Dense(OUTPUT_WINDOW)(x)            # une valeur par horizon J+1..J+OUTPUT_WINDOW
    m = keras.Model(inp, out, name=name)
    m.compile(optimizer=keras.optimizers.Adam(LR), loss="mse", metrics=["mae"])
    return m

for nm in MODEL_NAMES:
    print(f"{nm:6s} | params: {build_model(nm).count_params()/1e6:.3f} M")

## 7. Benchmark — entraînement des 3 modèles
Boucle identique pour chaque modèle : `model.fit` avec early stopping, réduction du
learning rate sur la validation et sauvegarde du meilleur checkpoint par modèle.

In [ ]:
CKPT_DIR   = ART_DIR / "checkpoints"; CKPT_DIR.mkdir(exist_ok=True)
MODELS_DIR = ART_DIR / "models";      MODELS_DIR.mkdir(exist_ok=True)

def train_keras(name, epochs=EPOCHS):
    m = build_model(name)
    es = EarlyStopping(monitor="val_loss", patience=PATIENCE,
                       restore_best_weights=True, verbose=0)
    cbs = [
        es,
        ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                          patience=max(2, PATIENCE // 2), min_lr=1e-6, verbose=0),
        # Reprise après coupure : état COMPLET (poids + optimiseur + epoch) par modèle,
        # repris automatiquement à l'epoch interrompue au redémarrage du notebook.
        keras.callbacks.BackupAndRestore(backup_dir=str(CKPT_DIR / f"backup_{name}")),
        keras.callbacks.ModelCheckpoint(str(CKPT_DIR / f"{name}_best.keras"),
                                        monitor="val_loss", save_best_only=True, verbose=0),
        # Historique persistant par epoch (append -> survit aux coupures)
        keras.callbacks.CSVLogger(str(CKPT_DIR / f"{name}_history.csv"), append=True),
    ]
    h = m.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=epochs,
              batch_size=BATCH_SIZE, callbacks=cbs, verbose=1)
    # Où l'entraînement s'est arrêté (epoch) + info de reprise, par modèle
    n_run = len(h.history["loss"])
    info = {"epochs_run": n_run,
            "best_epoch": int(np.argmin(h.history["val_loss"])) + 1,
            "early_stopped_epoch": int(es.stopped_epoch) or None,
            "backup_dir": str(CKPT_DIR / f"backup_{name}")}
    json.dump(info, open(CKPT_DIR / f"{name}_resume.json", "w"), indent=2)
    print(f"  [{name}] epochs effectuées: {n_run} | meilleure epoch: {info['best_epoch']}"
          f" | early_stop_epoch: {info['early_stopped_epoch']}")
    return m, h.history

def inv_y(arr):                                  # [N, OUTPUT_WINDOW] normalisé -> physique (°C)
    return scaler_y.inverse_transform(arr.reshape(-1, 1)).reshape(arr.shape)

def eval_metrics(p, t):                          # p, t en unités physiques, shape [N, OUTPUT_WINDOW]
    rmse = np.sqrt(((p - t) ** 2).mean(axis=0))  # [OUTPUT_WINDOW] (par horizon)
    mae  = np.abs(p - t).mean(axis=0)
    r2   = np.array([r2_score(t[:, h], p[:, h]) for h in range(t.shape[1])])
    return rmse, mae, r2

true = inv_y(y_te)
results, histories = {}, {}
for name in MODEL_NAMES:
    print(f"\n===== {name} =====")
    m, hist = train_keras(name)
    m.save(str(MODELS_DIR / f"{name}.keras"))    # on sauvegarde CHAQUE modèle entraîné
    pred = inv_y(m.predict(X_te, batch_size=BATCH_SIZE, verbose=0))
    rmse, mae, r2 = eval_metrics(pred, true)
    results[name] = {"model": m, "pred": pred, "rmse": rmse, "mae": mae, "r2": r2,
                     "n_params": m.count_params()}
    histories[name] = hist

# Meilleur modèle (RMSE moyen sur le test) -> exposé pour les cellules suivantes
best_name = min(results, key=lambda k: results[k]["rmse"].mean())
model, pred = results[best_name]["model"], results[best_name]["pred"]
print(f"\nMeilleur modèle (RMSE moyen test): {best_name}")

In [ ]:
# Courbes d'apprentissage (validation) — tous les modèles
plt.figure(figsize=(8, 4))
for name in MODEL_NAMES:
    plt.plot(histories[name]["val_loss"], label=f"{name} (val)")
plt.xlabel("epoch"); plt.ylabel("MSE (normalisé)")
plt.title("Courbes d'apprentissage — comparaison"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 7bis. Comparaison des modèles
Tableau récapitulatif (RMSE / MAE / R² moyens sur les horizons, en °C sur le jeu de
test) et comparaison visuelle du RMSE.

In [ ]:
# ── Tableau comparatif ──
rows = []
for name in MODEL_NAMES:
    r = results[name]
    rows.append({
        "modèle": name,
        "params(M)": round(r["n_params"] / 1e6, 3),
        "RMSE": round(float(r["rmse"].mean()), 3),
        "MAE":  round(float(r["mae"].mean()), 3),
        "R2":   round(float(r["r2"].mean()), 3),
    })
bench_df = pd.DataFrame(rows).set_index("modèle")
print(bench_df.to_string())
bench_df

In [ ]:
# ── Comparaison visuelle : RMSE / MAE / R² moyens par modèle ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, ylab in zip(axes, ["RMSE", "MAE", "R2"],
                         ["RMSE (°C)", "MAE (°C)", "R²"]):
    ax.bar(bench_df.index, bench_df[col].values, color=["#4C72B0", "#DD8452", "#55A868"])
    ax.set_title(f"{col} par modèle"); ax.set_ylabel(ylab)
    for i, v in enumerate(bench_df[col].values):
        ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
plt.suptitle("Comparaison des modèles temporels — heat_index", fontweight="bold")
plt.tight_layout(); plt.show()

## 7ter. Métriques par horizon de prévision
RMSE / MAE / R² en fonction de l'horizon `J+1 … J+OUTPUT_WINDOW`, une courbe par
modèle. L'erreur croît normalement avec l'horizon : prévoir J+7 est plus dur que J+1.

In [ ]:
H = OUTPUT_WINDOW
horizons = np.arange(1, H + 1)
panels = [("RMSE", "rmse"), ("MAE", "mae"), ("R²", "r2")]
fig, axes = plt.subplots(1, len(panels), figsize=(6 * len(panels), 4))
for ax, (label, key) in zip(axes, panels):
    for name in MODEL_NAMES:
        ax.plot(horizons, results[name][key], marker="o", label=name)
    ax.set_title(f"{label} par horizon — heat_index")
    ax.set_xlabel("horizon (jours, J+h)"); ax.set_ylabel(label)
    ax.set_xticks(horizons); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

## 8. Prédictions vs réel (meilleur modèle, J+1 sur le jeu de test)

In [ ]:
plt.figure(figsize=(13, 5))
n = min(400, pred.shape[0])
plt.plot(true[:n, 0], label="réel", lw=1)
plt.plot(pred[:n, 0], label=f"prévu J+1 ({best_name})", lw=1, alpha=0.8)
plt.title(f"heat_index — J+1 ({best_name}, test)"); plt.legend()
plt.xlabel("échantillon"); plt.ylabel("heat_index (°C)")
plt.tight_layout(); plt.show()

# Nuage prévu vs réel (tous horizons) pour le meilleur modèle
r2_best = float(results[best_name]["r2"].mean())
plt.figure(figsize=(6, 6))
plt.scatter(true.ravel(), pred.ravel(), s=4, alpha=0.2)
lims = [min(true.min(), pred.min()), max(true.max(), pred.max())]
plt.plot(lims, lims, "r--", lw=1)
plt.xlabel("réel (°C)"); plt.ylabel("prévu (°C)")
plt.title(f"heat_index — prévu vs réel ({best_name}, R²={r2_best:.3f})")
plt.tight_layout(); plt.show()

## 9. Sauvegarde des artefacts

In [ ]:
# NB : chaque modèle est déjà sauvegardé dans ART_DIR/models/ pendant le benchmark.
best = results[best_name]["model"]
best.save(str(ART_DIR / "best_model_heat_index.keras"))     # alias du MEILLEUR modèle
joblib.dump(scaler_X, ART_DIR / "scaler_X.pkl")
joblib.dump(scaler_y, ART_DIR / "scaler_y.pkl")
bench_df.to_csv(ART_DIR / "benchmark_metrics.csv")

meta = {
    "framework": "tensorflow", "best_model": best_name,
    "models": {name: {"path": f"models/{name}.keras",
                      "rmse_mean": round(float(results[name]["rmse"].mean()), 3),
                      "mae_mean":  round(float(results[name]["mae"].mean()), 3),
                      "r2_mean":   round(float(results[name]["r2"].mean()), 3),
                      "params":    int(results[name]["n_params"])}
               for name in MODEL_NAMES},
    "features": FEATURES, "target": TARGET,
    "scaler": "minmax", "feature_range": [0, 1],
    "month_filter": [MONTH_START, MONTH_END],
    "split": {"train": TRAIN_RATIO, "val": VAL_RATIO, "test": TEST_RATIO,
              "scheme": "per-station chronological"},
    "input_window": INPUT_WINDOW, "output_window": OUTPUT_WINDOW,
    "hidden": HIDDEN, "n_layers": N_LAYERS, "dropout": DROPOUT,
    "dataset": "data/cleandata_gsod_by_station",
}
json.dump(meta, open(ART_DIR / "benchmark_meta.json", "w"), indent=2)
print("Modèles sauvegardés ->", MODELS_DIR, "(", ", ".join(MODEL_NAMES), ")")
print("meilleur alias:", ART_DIR / "best_model_heat_index.keras",
      "| scalers | benchmark_metrics.csv | benchmark_meta.json")